# Using NABOPB for a PDE

A hands-on walkthrough of `NABOPB_for_PDE`, the entry point this repository uses in
`pde_applications/`. One example from start to finish: install, write the solver in the shape
the function expects, call it, read the result, use it. No theory. The companion
`tutorial_usage.ipynb` covers the plain `NABOPB` call; this notebook assumes you have seen it.

The example is the Poisson equation in one space dimension with a parameterised right-hand
side. What comes out is an approximation of the **solution operator**: a single formula that
maps any right-hand side and any point $x$ to the solution value $u(x)$, learned from a limited
number of PDE solves.

## 0. Before you start

Install as in `tutorial_usage.ipynb`, step 0: clone, create `.venv`, then

```
pip install -r requirements.txt ipykernel ipywidgets
```

and select `.venv` as the kernel. This notebook needs the compressed-sensing part of the
package: `NABOPB_for_PDE` only works with the `_OMP+`, `_COSAMP+` and `_SRLasso+` methods,
which run through KeOps. KeOps compiles small C++ kernels the first time a method is used.
That takes a minute or two, once, and needs a working C++ compiler. It also prints a few
warnings about CUDA and OpenMP on import; they are expected on a machine without a GPU.

In [1]:
import sys, time, warnings
from pathlib import Path
import numpy as np
from collections import Counter

REPO = Path.cwd() if (Path.cwd() / "NABOPB_for_PDE.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO))

with warnings.catch_warnings():
    warnings.simplefilter("ignore", SyntaxWarning)      # one upstream file has LaTeX in plain strings
    from NABOPB_for_PDE import NABOPB_for_PDE

print(f"numpy {np.__version__}   repo {REPO.name}/   ready")

[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.
[KeOps] Warning : OpenMP library not found, it must be downloaded through Homebrew for apple Silicon chips
[KeOps] Warning : OpenMP support is not available. Disabling OpenMP.


numpy 2.2.6   repo nabopb/   ready


## 1. The problem

$$-u''(x) = f(x) \quad \text{on } [0, 1], \qquad u(0) = u(1) = 0,$$

with a right-hand side described by nine numbers $a_{-4}, \dots, a_4$:

$$f(x) = a_0 + \sum_{k \neq 0} a_k \cos(2\pi k x).$$

Each choice of the nine numbers is a different problem with a different solution. We want one
object that answers all of them: the map $(x, a_{-4}, \dots, a_4) \mapsto u(x)$. That is a
function of **ten variables**, one spatial and nine parameters, and it is what NABOPB will
approximate.

For this equation the solution is known in closed form, which keeps every cell below fast. The
closed form plays the role of "the solver"; step 6 swaps in a real numerical one.

In [2]:
K = np.arange(-4, 5)                                   # the nine modes of the right-hand side

def solve_poisson(x, a):
    """u(x) for right-hand side coefficients a.  x: (n,) in [0, 1];  a: (n, 9), one row per x."""
    k = K.astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):              # the k = 0 term is handled separately
        terms = a / (4 * np.pi**2 * k**2) * (np.cos(2 * np.pi * np.outer(x, k)) - 1)
    return a[:, 4] / 2 * x * (1 - x) + np.nansum(terms, axis=1)

# sanity check: f = 1 (only a_0 = 1) gives u = x(1 - x)/2, so u(0) = u(1) = 0 and u(1/2) = 1/8
a_test = np.zeros((3, 9)); a_test[:, 4] = 1.0
for xv, u in zip([0.0, 0.5, 1.0], solve_poisson(np.array([0.0, 0.5, 1.0]), a_test)):
    print(f"   u({xv}) = {u:.4f}")

   u(0.0) = 0.0000
   u(0.5) = 0.1250
   u(1.0) = 0.0000


## 2. Provide the solver in the shape `NABOPB_for_PDE` expects

The plain `NABOPB` call takes a function of one array, `f(x)` with `x` of shape
`(num_points, d)`. `NABOPB_for_PDE` splits that into two arguments and adds a third:

> `handle(x_pde, params, blocksize)` receives
> `x_pde` of shape `(n, pde_dims)`, the points in the PDE's own variables (here just $x$);
> `params` of shape `(m, d - pde_dims)`, one row per right-hand side;
> and `blocksize`, with `n = m * blocksize`: rows `0` to `blocksize - 1` of `x_pde` belong to
> `params[0]`, the next `blocksize` rows to `params[1]`, and so on.
> It returns `n` values, one per row of `x_pde`.

The split is the point of this function. Solving a PDE gives you the solution everywhere, so
one solve per parameter row can serve all `blocksize` points at once. The algorithm groups its
requests so that happens as often as possible.

Everything lives on $[-1, 1]$, because we use a Chebyshev basis. The handle maps $x$ back to
$[0, 1]$ and uses the parameters as they come.

In [3]:
def handle(x_pde, params, blocksize=1):
    a = np.repeat(params, blocksize, axis=0)[: x_pde.shape[0]]    # parameter row i for block i
    x = (x_pde[:, 0] + 1) / 2                                      # [-1, 1] -> [0, 1]
    return solve_poisson(x, a)

# what a call looks like: 2 right-hand sides, 3 points each
x_demo = np.array([[-1.0], [0.0], [1.0], [-1.0], [0.0], [1.0]])
a_demo = np.zeros((2, 9)); a_demo[0, 4] = 1.0; a_demo[1, 4] = 2.0
print("x_pde shape", x_demo.shape, " params shape", a_demo.shape, " blocksize 3")
print("returns     ", np.round(handle(x_demo, a_demo, blocksize=3), 4), " (f = 1 gives u(1/2) = 0.125, f = 2 gives 0.25)")

x_pde shape (6, 1)  params shape (2, 9)  blocksize 3
returns      [0.    0.125 0.    0.    0.25  0.   ]  (f = 1 gives u(1/2) = 0.125, f = 2 gives 0.25)


## 3. Call `NABOPB_for_PDE`

```python
result = NABOPB_for_PDE(handle, pde_dims, d, basis_flag, gamma, diminc, sparsity_s)
```

Compared with `NABOPB`, one argument is new and one is restricted:

- `pde_dims` is the number of variables that belong to the PDE itself, here `1` for $x$. The
  remaining `d - pde_dims` are parameters. The PDE variables must come first in the ordering.
- `basis_flag` has to be one of the `+` methods: `Cheby_OMP+`, `Cheby_COSAMP+`,
  `Cheby_SRLasso+`, or the `Fourier_` equivalents. The lattice methods stop with a reshape
  error inside this function.

A further optional argument, `pde_disc` (default `3`), sets how many extra points per PDE
variable each solve is evaluated at, so `pde_disc ** pde_dims` extra points per solve. Larger
values extract more from every solve.

The call prints a progress line per step and, for the `+` methods, the internal residuals of
the sparse solver. They are hidden below with `redirect_stdout`; drop that line to see them.

`gamma` is as before: a box of half-width `N`, non-negative because Chebyshev indices have no
sign. **Parameters are reduced so the cell runs in a second.** `pde_applications/demo_pois_1d_f.py`
uses `s = 1000` and `N = 64`.

In [4]:
d, pde_dims = 10, 1
gamma = {"type": "full", "N": 8, "w": 1, "sgn": "non-negative"}
diminc = {"type": "default"}

import contextlib, io

np.random.seed(0)
t0 = time.time()
with contextlib.redirect_stdout(io.StringIO()):           # hides progress lines and solver residuals
    result = NABOPB_for_PDE(handle, pde_dims, d, "Cheby_OMP+", gamma, diminc, sparsity_s=40)
print(f"done in {time.time() - t0:.1f}s")

done in 0.5s


## 4. Read the result

The same dictionary as before: `index` holds the detected indices, one row per coefficient,
with the $x$ degree in column 0 and the degree in $a_{-4}, \dots, a_4$ in columns 1 to 9;
`val` holds the coefficients.

One field changes meaning. `sample_sizes` counts **PDE solves**, not point evaluations: every
time the algorithm asked for a new right-hand side, that is one solve, however many points it
was evaluated at.

In [5]:
index, val = result["index"], np.real(result["val"])
solves = int(np.sum(result["sample_sizes"]))

print(f"{index.shape[0]} coefficients from {solves:,} PDE solves")
print("solves per step:", [int(v) for v in result["sample_sizes"]])

deg_x = index[:, 0]
deg_a = index[:, 1:].sum(axis=1)
print("\ndegree in x            :", dict(sorted(Counter(deg_x.tolist()).items())))
print("total degree in a      :", dict(sorted(Counter(deg_a.tolist()).items())))
print(f"largest |coefficient|  : linear in a {np.abs(val[deg_a == 1]).max():.4f},  "
      f"nonlinear in a {np.abs(val[deg_a > 1]).max():.4f}")

print("\nthe six largest terms   (x degree, which a_k, coefficient)")
for i in np.argsort(-np.abs(val))[:6]:
    which = ", ".join(f"a_{K[j]}" for j in np.flatnonzero(index[i, 1:]))
    print(f"   {index[i, 0]:>2}   {which:<8} {val[i]:>10.6f}")

40 coefficients from 1,402 PDE solves
solves per step: [722, 80, 80, 80, 80, 80, 80, 80, 80, 40]

degree in x            : {0: 9, 2: 12, 4: 8, 6: 5, 8: 6}
total degree in a      : {1: 32, 2: 4, 3: 4}
largest |coefficient|  : linear in a 0.0440,  nonlinear in a 0.0005

the six largest terms   (x degree, which a_k, coefficient)
    0   a_0        0.044047
    2   a_0       -0.031488
    0   a_1       -0.012496
    0   a_-1      -0.012392
    2   a_1        0.012222
    2   a_-1       0.012121


The first entry of `solves per step` covers the ten one-dimensional detections. Nine of them
are parameter directions and cost 80 solves each; the $x$ direction cost **two**, because a
single solve already gives $u$ at every $x$ the algorithm wanted. That is the saving this
function is built around, and it is why the total is about 1,400 solves rather than the
13,000 or so point evaluations the plain `NABOPB` call needs for the same problem.

Two things in the rest of the output are facts about the equation that the algorithm found on
its own. Most detected terms, 32 of 40, are of **degree 1 in the parameters**, and the eight
that are not carry coefficients about a hundred times smaller. The Poisson equation is linear:
doubling $f$ doubles $u$. The approximation reflects that.

Every detected $x$ degree is **even**. The right-hand side is built from cosines, which are
symmetric about $x = 1/2$, so the solution is too, and on $[-1, 1]$ a symmetric function has
only even Chebyshev terms.

## 5. Use the learned operator

`index` and `val` define a function of all ten variables. Evaluating it is one matrix product,
the same helper as in `tutorial_usage.ipynb` step 6. The input must be on $[-1, 1]$ in every
variable, so map $x$ before calling.

In [6]:
def evaluate_chebyshev(index, val, X):
    """Evaluate the approximation at points X of shape (num_points, d) on [-1, 1]^d."""
    T = np.cos(index[None, :, :] * np.arccos(X)[:, None, :])
    scale = np.sqrt(2.0) ** np.count_nonzero(index, axis=1)
    return np.real(np.prod(T, axis=2) @ (val * scale))

# accuracy over many new (x, a) combinations, drawn the Chebyshev way
np.random.seed(7)
X_new = np.sin(np.pi * (np.random.rand(20000, d) - 0.5))
u_true = handle(X_new[:, :1], X_new[:, 1:], blocksize=1)
u_learned = evaluate_chebyshev(index, result["val"], X_new)
print(f"relative L2 error at 20,000 new (x, a) points: "
      f"{np.linalg.norm(u_learned - u_true) / np.linalg.norm(u_true):.3f}")

# one fixed right-hand side, the solution along x
np.random.seed(11)
a_fixed = np.random.rand(9) * 2 - 1
xs = np.array([0.0, 0.25, 0.5, 0.8, 1.0])
X_line = np.hstack([2 * xs[:, None] - 1, np.tile(a_fixed, (5, 1))])
print(f"\none right-hand side, u along x    {'x':>5}{'learned':>10}{'true':>10}")
for xv, ul, ut in zip(xs, evaluate_chebyshev(index, result["val"], X_line), solve_poisson(xs, np.tile(a_fixed, (5, 1)))):
    print(f"{'':<34}{xv:>5}{ul:>10.5f}{ut:>10.5f}")

relative L2 error at 20,000 new (x, a) points: 0.029

one right-hand side, u along x        x   learned      true
                                    0.0   0.00053   0.00000
                                   0.25  -0.00927  -0.00957
                                    0.5  -0.03737  -0.03571
                                    0.8  -0.00400  -0.00337
                                    1.0   0.00053   0.00000


An error of a few percent from 40 coefficients and about 1,400 solves. The boundary values are
not exactly zero because nothing told the approximation about the boundary condition; it
learned the operator from samples alone.

## 6. Plug in a real solver

The closed form stood in for a solver. Here is the same problem solved numerically with
`scipy.integrate.solve_bvp`, as `pde_applications/demo_pois_1d_s.py` does. The handle is
written the way a real solver should be used: **one solve per parameter row**, then evaluated
at that row's block of points. Nothing else changes.

In [7]:
from scipy.integrate import solve_bvp

def rhs(x, a):
    return a[4] + sum(a[j] * np.cos(2 * np.pi * K[j] * x) for j in range(9) if K[j] != 0)

def solve_poisson_numeric(x, a, tol=1e-6):
    """Solve -u'' = f(.; a), u(0) = u(1) = 0 numerically; return u at the points x (in [0, 1])."""
    mesh = np.linspace(0, 1, 50)
    sol = solve_bvp(lambda t, y: np.vstack([y[1], -rhs(t, a)]),
                    lambda ya, yb: np.array([ya[0], yb[0]]), mesh, np.zeros((2, mesh.size)), tol=tol)
    return sol.sol(x)[0]

def handle_numeric(x_pde, params, blocksize=1):
    out = np.empty(x_pde.shape[0])
    for i, a in enumerate(params):                                  # one solve per right-hand side ...
        block = slice(i * blocksize, (i + 1) * blocksize)
        out[block] = solve_poisson_numeric((x_pde[block, 0] + 1) / 2, a)   # ... evaluated at its block
    return out

t0 = time.time()
u_num = handle_numeric(X_line[:, :1], a_fixed[None, :], blocksize=5)
per_solve = time.time() - t0
print(f"one solve, five points: {1000 * per_solve:.0f} ms;  max difference to the closed form: "
      f"{np.max(np.abs(u_num - solve_poisson(xs, np.tile(a_fixed, (5, 1))))):.1e}")
total = solves * per_solve
print(f"the {solves:,} solves of step 3 would take about {total:.0f} seconds with this solver, "
      f"ten times that with one solve per sample point")

one solve, five points: 55 ms;  max difference to the closed form: 4.1e-10
the 1,402 solves of step 3 would take about 77 seconds with this solver, ten times that with one solve per sample point


That last line is why `NABOPB_for_PDE` exists. Calling the solver through plain `NABOPB` means
one solve per sample point, about 13,000 here for the same accuracy, ten times more. For this
small problem both are affordable. The heat and Burgers examples in `pde_applications/` cost
about a second per solve, and there the factor of ten is the difference between twenty
minutes and three hours. Those scripts also run the solver through `multiprocessing.Pool`,
which this notebook leaves out for clarity.

To use your own solver: write it in the shape of `handle_numeric`, map the PDE variables from
$[-1, 1]$ to your domain, pass `pde_dims` for the number of PDE variables, and pick one of the
`+` methods. Start with a small `sparsity_s` and increase it.